# Robustness Certification

## Robustness Certification

Robustness certification answers: for all inputs within an ε-ball of x, does the network output the same class?

**L∞ certification**: The most common. Input perturbations bounded by max pixel change ε.
**L2 certification**: Euclidean ball perturbations.

**Certified accuracy** measures what fraction of test examples can be provably certified robust, as opposed to empirically robust (survives PGD attacks but not formally verified).

The certified accuracy is always ≤ empirical accuracy under attack. The gap measures verification conservatism.

In [ ]:
import numpy as np

np.random.seed(42)

def sigmoid(z): return 1 / (1 + np.exp(-np.clip(z, -500, 500)))

# Train a small classifier
N, D = 400, 8
X = np.random.randn(N, D)
y = (X[:, 0] + X[:, 1] > 0).astype(float)

W1 = np.random.randn(16, D) * 0.3
b1 = np.zeros(16)
W2 = np.random.randn(1, 16) * 0.3
b2 = np.zeros(1)

# Quick training
lr = 0.05
for _ in range(300):
    h1 = np.maximum(0, X @ W1.T + b1)
    out = sigmoid(h1 @ W2.T + b2)[:, 0]
    err = out - y
    dW2 = (err[:, None] * out[:, None] * (1-out[:, None]) * h1).mean(0, keepdims=True)
    dh1 = np.outer(err * out * (1-out), W2[0])
    dh1_pre = dh1 * (h1 > 0)
    dW1 = dh1_pre.T @ X / N
    W1 -= lr * dW1; W2 -= lr * dW2

# Compute certified accuracy via IBP
def ibp_bounds(x, eps, W1, b1, W2, b2):
    x_lo, x_hi = x - eps, x + eps
    W1p, W1n = np.maximum(0, W1), np.minimum(0, W1)
    pre_lo = W1p @ x_lo + W1n @ x_hi + b1
    pre_hi = W1p @ x_hi + W1n @ x_lo + b1
    h_lo, h_hi = np.maximum(0, pre_lo), np.maximum(0, pre_hi)
    W2p, W2n = np.maximum(0, W2[0]), np.minimum(0, W2[0])
    out_lo = W2p @ h_lo + W2n @ h_hi + b2[0]
    out_hi = W2p @ h_hi + W2n @ h_lo + b2[0]
    return out_lo, out_hi

X_test, y_test = X[300:], y[300:]

print('Certified vs Empirical Robustness:')
print(f'{'eps':>6} {'Nominal':>8} {'Empirical':>10} {'Certified':>10}')
print('-' * 40)

for eps in [0.0, 0.05, 0.1, 0.2, 0.5]:
    nominal = 0
    empirical = 0
    certified = 0
    
    for x, yt in zip(X_test, y_test):
        h = np.maximum(0, x @ W1.T + b1)
        p = sigmoid((h @ W2.T + b2)[0, 0])
        pred = (p > 0.5)
        correct = (int(pred) == int(yt))
        nominal += correct
        
        if eps == 0:
            empirical += correct
            certified += correct
        else:
            out_lo, out_hi = ibp_bounds(x, eps, W1, b1, W2, b2)
            if yt == 1:
                cert = (out_lo > 0)
            else:
                cert = (out_hi < 0)
            certified += int(cert)
            empirical += correct  # simplified
    
    n = len(X_test)
    print(f'{eps:6.2f} {nominal/n:8.3f} {empirical/n:10.3f} {certified/n:10.3f}')
